# Hosting Strands Agents with OpenAI models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. 

We will focus on a Strands Agents with OpenAI model example. For Strands Agents with Amazon Bedrock model check [here](../01-strands-with-bedrock-model) and 
for LangGraph with Amazon Bedrock model check [here](../02-langgraph-with-bedrock-model)


### Tutorial Details

| Information         | Details                                                                  |
|:--------------------|:-------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                           |
| Agent type          | Single                                                                   |
| Agentic Framework   | Strands Agents                                                           |
| LLM model           | GPT 4.1 mini                                                             |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and OpenAI Model |
| Tutorial vertical   | Cross-vertical                                                           |
| Example complexity  | Easy                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                             |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using OpenAI models
* Using Strands Agents


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker running

In [ ]:
!pip install -r requirements.txt

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import boto3
import sys
import os
import json
import time

# Get the current notebook's directory
current_dir = os.path.dirname(
    os.path.abspath("__file__" if "__file__" in globals() else ".")
)

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.join(utils_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role

In [ ]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)
iam_client = boto3.client("iam")
codebuild_client = boto3.client("codebuild")

agent_name = "strands_agents_openai"

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

The architecture here will look as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os

os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_open_ai(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_open_ai(json.loads(args.payload))
    print(response)

#### Invoking local agent

In [ ]:
!python strands_agents_openai.py '{"prompt": "What is the weather now?"}'

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with OpenAI model
Let's start with our Strands Agent using the GPT 4.1 mini model. All the others will work exactly the same.

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)


@app.entrypoint
def strands_agent_open_ai(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Creating runtime role

Before starting, let's create an IAM role for our AgentCore Runtime. We will do so using the utils function pre-developed for you.

In [ ]:
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
print(f"Using AWS region: {region}")

required_files = ["strands_agents_openai.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

print("Configuring AgentCore Runtime...")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_agents_openai.py",
    execution_role=agentcore_iam_role["Role"]["Arn"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
)
response

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

When using the `agentcore_runtime.launch()` method, you can optionally pass `use_codebuild=True` to use [AWS CodeBuild](https://aws.amazon.com/codebuild/) instead of local Docker to build and push the image to Amazon ECR.

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch(use_codebuild=True)

#### If you want to build the image using local Docker, run this command instead ####
# launch_result = agentcore_runtime.launch()print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
print(f"Initial status: {status}")

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    print("✓ AgentCore Runtime is READY!")
else:
    print(f"⚠ AgentCore Runtime status: {status}")

print(f"Final status: {status}")

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "Hi, what can you do?"})

print(invoke_response)

### Processing invocation results

We can now process our invocation results to include it in an application

In [ ]:
response_text = json.loads(invoke_response["response"][0].decode("utf-8"))

print(response_text)

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
agent_arn = launch_result.agent_arn

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How much is 2X2?"}),
)
# Process and print the response
if "text/event-stream" in boto3_response.get("contentType", ""):
  
    # Handle streaming response
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=10):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    print("\nComplete response:", "\n".join(content))

elif boto3_response.get("contentType") == "application/json":
    # Handle standard JSON response
    content = []
    for chunk in boto3_response.get("response", []):
        content.append(chunk.decode('utf-8'))
    print(json.loads(''.join(content)))
  
else:
    # Print raw response for other content types
    print(boto3_response)

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
print(f"ECR Repository URI: {launch_result.ecr_uri}")
print(f"Agent ID: {launch_result.agent_id}")
print(f"ECR Repository Name: {launch_result.ecr_uri.split('/')[1]}")
print(f"CodeBuild Project Name: {launch_result.codebuild_id.split(':')[0]}")

In [ ]:
# print("🗑️  Starting cleanup process...")

# print("Deleting AgentCore Runtime...")
# try:
#     runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#         agentRuntimeId=launch_result.agent_id,
#     )
#     print(f"Deleted agent runtime: {launch_result.agent_id}")
# except Exception as e:
#     print(f"Agent runtime {launch_result.agent_id} not found or already deleted: {e}")
#     print("You may need to manually clean up some resources.")

# print("Deleting ECR repository...")
# try:
#     response = ecr_client.delete_repository(
#         repositoryName=launch_result.ecr_uri.split("/")[1], force=True
#     )
#     print(f"Deleted ECR repository: {launch_result.ecr_uri.split('/')[1]}")
# except Exception as e:
#     print(
#         f"ECR repository {launch_result.ecr_uri.split('/')[1]} not found or already deleted: {e}"
#     )
#     print("You may need to manually clean up some resources.")

# print("Deleting IAM role policies...")
# try:
#     policies = iam_client.list_role_policies(
#         RoleName=agentcore_iam_role["Role"]["RoleName"], MaxItems=100
#     )

#     for policy_name in policies["PolicyNames"]:
#         iam_client.delete_role_policy(
#             RoleName=agentcore_iam_role["Role"]["RoleName"], PolicyName=policy_name
#         )

#     iam_response = iam_client.delete_role(
#         RoleName=agentcore_iam_role["Role"]["RoleName"]
#     )
#     print(f"Deleted IAM role: {agentcore_iam_role['Role']['RoleName']}")
# except Exception as e:
#     print(
#         f"IAM role {agentcore_iam_role['Role']['RoleName']} not found or already deleted: {e}"
#     )
#     print("You may need to manually clean up some resources.")

# try:
#     response = codebuild_client.delete_project(
#         name=launch_result.codebuild_id.split(":")[0]
#     )
#     print(f"Deleted CodeBuild project: {launch_result.codebuild_id.split(':')[0]}")
# except Exception as e:
#     print(
#         f"CodeBuild project {launch_result.codebuild_id.split(':')[0]} not found or already deleted: {e}"
#     )
#     print("You may need to manually clean up some resources.")

# print("\n✅ Cleanup completed successfully!")

# Congratulations!